In [4]:
# ============================================================
# CTG FETAL DISTRESS DETECTION
# MODEL COMPARISON + FINAL MODEL SELECTION
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. LOAD TEST DATA
# ============================================================

print("========================================")
print("       CTG MODEL COMPARISON")
print("========================================")

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("\n===== TEST DATA =====")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


# ============================================================
# 2. LOAD TRAINED MODELS
# ============================================================

rf_model = joblib.load("../models/random_forest.pkl")
xgb_model = joblib.load("../models/xgboost.pkl")

print("\n===== MODELS LOADED =====")
print("Random Forest : OK")
print("XGBoost       : OK")


# ============================================================
# 3. MAKE PREDICTIONS
# ============================================================

rf_pred = rf_model.predict(X_test)

# XGBoost was trained using:
# 1 -> 0
# 2 -> 1
# 3 -> 2
#
# Convert predictions back:
# 0 -> 1
# 1 -> 2
# 2 -> 3

xgb_pred = xgb_model.predict(X_test)
xgb_pred = xgb_pred.astype(int) + 1


# ============================================================
# 4. CHECK PREDICTIONS
# ============================================================

print("\n===== PREDICTION CHECK =====")

print("Actual classes:")
print(np.unique(y_test))

print("\nRandom Forest predictions:")
print(np.unique(rf_pred))

print("\nRandom Forest prediction counts:")
print(
    pd.Series(rf_pred)
    .value_counts()
    .sort_index()
)

print("\nXGBoost predictions:")
print(np.unique(xgb_pred))

print("\nXGBoost prediction counts:")
print(
    pd.Series(xgb_pred)
    .value_counts()
    .sort_index()
)


# ============================================================
# 5. CALCULATE ACCURACY
# ============================================================

rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)


# ============================================================
# 6. CALCULATE MACRO F1
# ============================================================

rf_f1 = f1_score(
    y_test,
    rf_pred,
    average="macro"
)

xgb_f1 = f1_score(
    y_test,
    xgb_pred,
    average="macro"
)


# ============================================================
# 7. MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        rf_accuracy,
        xgb_accuracy
    ],
    "Macro F1": [
        rf_f1,
        xgb_f1
    ]
})


# ============================================================
# 8. PRINT CLEAN COMPARISON
# ============================================================

print("\n===== MODEL COMPARISON =====\n")

print(
    f"{'Model':<20}"
    f"{'Accuracy':<12}"
    f"{'Macro F1':<12}"
)

print(
    f"{'Random Forest':<20}"
    f"{rf_accuracy:.4f}      "
    f"{rf_f1:.4f}"
)

print(
    f"{'XGBoost':<20}"
    f"{xgb_accuracy:.4f}      "
    f"{xgb_f1:.4f}"
)


# ============================================================
# 9. RANDOM FOREST DETAILED RESULTS
# ============================================================

print("\n====================================")
print("RANDOM FOREST RESULTS")
print("====================================")

print(
    classification_report(
        y_test,
        rf_pred,
        target_names=[
            "Normal",
            "Suspect",
            "Pathological"
        ]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        rf_pred
    )
)


# ============================================================
# 10. XGBOOST DETAILED RESULTS
# ============================================================

print("\n====================================")
print("XGBOOST RESULTS")
print("====================================")

print(
    classification_report(
        y_test,
        xgb_pred,
        target_names=[
            "Normal",
            "Suspect",
            "Pathological"
        ]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        xgb_pred
    )
)


# ============================================================
# 11. SELECT BEST MODEL
# ============================================================
#
# Macro F1 is used as the primary selection metric
# because the dataset has imbalanced classes.
#
# If Macro F1 is equal, Accuracy is used as tie-breaker.
# ============================================================

if (
    rf_f1 > xgb_f1
    or (
        rf_f1 == xgb_f1
        and rf_accuracy >= xgb_accuracy
    )
):
    best_model = "Random Forest"
    best_accuracy = rf_accuracy
    best_f1 = rf_f1

else:
    best_model = "XGBoost"
    best_accuracy = xgb_accuracy
    best_f1 = xgb_f1


# ============================================================
# 12. FINAL MODEL
# ============================================================

print("\n====================================")
print("FINAL MODEL")
print("====================================")

print(f"Best Model : {best_model}")
print(f"Accuracy   : {best_accuracy:.4f}")
print(f"Macro F1   : {best_f1:.4f}")


# ============================================================
# 13. SAVE MODEL COMPARISON
# ============================================================

comparison.to_csv(
    "../results/model_comparison.csv",
    index=False
)


# ============================================================
# 14. SAVE FINAL MODEL INFORMATION
# ============================================================

final_result = pd.DataFrame({
    "Final Model": [best_model],
    "Accuracy": [best_accuracy],
    "Macro F1": [best_f1]
})

final_result.to_csv(
    "../results/final_model.csv",
    index=False
)


# ============================================================
# 15. COMPLETE
# ============================================================

print("\n========================================")
print("        COMPARISON COMPLETE")
print("========================================")

print("\nSaved files:")
print("✓ ../results/model_comparison.csv")
print("✓ ../results/final_model.csv")

print("\n🏆 FINAL MODEL:", best_model)

       CTG MODEL COMPARISON

===== TEST DATA =====
X_test: (424, 41)
y_test: (424,)

===== MODELS LOADED =====
Random Forest : OK
XGBoost       : OK

===== PREDICTION CHECK =====
Actual classes:
[1. 2. 3.]

Random Forest predictions:
[1. 2. 3.]

Random Forest prediction counts:
1.0    332
2.0     57
3.0     35
Name: count, dtype: int64

XGBoost predictions:
[1 2 3]

XGBoost prediction counts:
1    333
2     56
3     35
Name: count, dtype: int64

===== MODEL COMPARISON =====

Model               Accuracy    Macro F1    
Random Forest       0.9953      0.9932
XGBoost             0.9929      0.9898

RANDOM FOREST RESULTS
              precision    recall  f1-score   support

      Normal       0.99      1.00      1.00       330
     Suspect       1.00      0.97      0.98        59
Pathological       1.00      1.00      1.00        35

    accuracy                           1.00       424
   macro avg       1.00      0.99      0.99       424
weighted avg       1.00      1.00      1.00     